# Demand Prediction Agent

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
df = pd.read_csv(
    r"F:\EV_TARIFF_OPTIMIZATION\data\cleaned_acn_data.csv"
)

df.head()

,_id,clusterID,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,timezone,userID,userInputs,session_duration_hr,charging_duration_hr,hour,day_of_week,is_weekend,fixed_tariff,fixed_revenue
0,5e210e1ff9af8b57bb4f5500,39,2020-01-01 02:12:11,2020-01-02 00:05:40,2020-01-01 06:55:27,15.813,2_39_139_28_2020-01-01 02:12:11.015844,2,CA-303,2-39-139-28,America/Los_Angeles,838.0,"[{'WhPerMile': 600, 'kWhRequested': 36.0, 'mil...",21.891389,4.721111,2,Wednesday,0,15,237.195
1,5e225f9ff9af8b5c26d21715,39,2020-01-01 09:42:14,2020-01-02 02:25:40,2020-01-01 14:42:11,32.020,2_39_131_30_2020-01-01 09:42:14.259248,2,CA-305,2-39-131-30,America/Los_Angeles,67.0,"[{'WhPerMile': 250, 'kWhRequested': 60.0, 'mil...",16.723889,4.999167,9,Wednesday,0,15,480.300
2,5e225f9ff9af8b5c26d21716,39,2020-01-01 18:10:34,2020-01-01 21:05:40,2020-01-01 19:22:56,2.328,2_39_127_19_2020-01-01 18:10:34.057445,2,CA-309,2-39-127-19,America/Los_Angeles,710.0,"[{'WhPerMile': 261, 'kWhRequested': 7.83, 'mil...",2.918333,1.206111,18,Wednesday,0,15,34.920
3,5e225f9ff9af8b5c26d21717,39,2020-01-01 19:44:51,2020-01-02 01:23:37,2020-01-01 22:43:57,19.868,2_39_79_377_2020-01-01 19:44:51.127414,2,CA-325,2-39-79-377,America/Los_Angeles,248.0,"[{'WhPerMile': 250, 'kWhRequested': 20.0, 'mil...",5.646111,2.985000,19,Wednesday,0,15,298.020
4,5e225f9ff9af8b5c26d21718,39,2020-01-02 01:12:29,2020-01-02 04:38:39,2020-01-02 03:11:48,8.336,2_39_126_20_2020-01-02 01:12:28.778216,2,CA-310,2-39-126-20,America/Los_Angeles,1099.0,"[{'WhPerMile': 400, 'kWhRequested': 24.0, 'mil...",3.436111,1.988611,1,Thursday,0,15,125.040


In [3]:
df.columns

Index(['_id', 'clusterID', 'connectionTime', 'disconnectTime',
       'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID',
       'stationID', 'timezone', 'userID', 'userInputs', 'session_duration_hr',
       'charging_duration_hr', 'hour', 'day_of_week', 'is_weekend',
       'fixed_tariff', 'fixed_revenue'],
      dtype='object')

In [4]:
X = df[['hour', 'is_weekend']]

y = df['kWhDelivered']

In [9]:
df[['kWhDelivered']].describe()

,kWhDelivered
count,1782.000000
mean,7.617972
std,8.375035
min,0.515000
25%,1.866250
50%,5.094500
75%,10.553500
max,53.925000


In [10]:
df[['hour','is_weekend','kWhDelivered']].head()

,hour,is_weekend,kWhDelivered
0,2,0,15.813
1,9,0,32.020
2,18,0,2.328
3,19,0,19.868
4,1,0,8.336


In [11]:
hourly_df = (
    df.groupby(['hour', 'is_weekend'])
      .agg(
          total_kwh=('kWhDelivered', 'sum'),
          sessions=('kWhDelivered', 'count')
      )
      .reset_index()
)

hourly_df.head()

,hour,is_weekend,total_kwh,sessions
0,0,0,285.890231,45
1,0,1,81.694395,16
2,1,0,226.401000,39
3,1,1,107.408000,12
4,2,0,273.027000,33


In [12]:
hourly_df.shape

(42, 4)

In [13]:
hourly_df.head(10)

,hour,is_weekend,total_kwh,sessions
0,0,0,285.890231,45
1,0,1,81.694395,16
2,1,0,226.401000,39
3,1,1,107.408000,12
4,2,0,273.027000,33
5,2,1,58.008000,12
6,3,0,185.209000,28
7,3,1,175.262000,11
8,4,0,139.056000,24
9,4,1,3.909000,2


In [14]:
X = hourly_df[['hour', 'is_weekend']]
y = hourly_df['total_kwh']

## Baseline Session-Level Model

This initial model was trained on individual charging sessions and achieved limited predictive performance (R² = 0.21). To improve forecasting accuracy, demand was aggregated at the hourly level and a second model was developed.

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [16]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [17]:
y_pred = model.predict(X_test)

In [18]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 485.61102376826955
RMSE: 875.9852103121901
R2 Score: 0.21230088205672237


## Improved Hourly Demand Prediction Model

In [19]:
df['connectionTime'] = pd.to_datetime(df['connectionTime'])

df['day_num'] = df['connectionTime'].dt.dayofweek

In [20]:
hourly_df2 = (
    df.groupby(['hour', 'is_weekend', 'day_num'])
      .agg(
          total_kwh=('kWhDelivered', 'sum'),
          sessions=('kWhDelivered', 'count'),
          avg_session_kwh=('kWhDelivered', 'mean')
      )
      .reset_index()
)

hourly_df2.head()

,hour,is_weekend,day_num,total_kwh,sessions,avg_session_kwh
0,0,0,0,39.583000,5,7.916600
1,0,0,1,91.239000,8,11.404875
2,0,0,2,59.095000,12,4.924583
3,0,0,3,31.861000,6,5.310167
4,0,0,4,64.112231,14,4.579445


In [21]:
X = hourly_df2[
    [
        'hour',
        'is_weekend',
        'day_num',
        'sessions',
        'avg_session_kwh'
    ]
]

y = hourly_df2['total_kwh']

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [23]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [24]:
y_pred = model.predict(X_test)

In [25]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 18.139533661825258
RMSE: 37.31266534813892
R2 Score: 0.9594990426203


In [30]:
import joblib
joblib.dump(
    model,
    r"F:\EV_Tariff_Optimization\output\demand_model.pkl"
)

['F:\\EV_Tariff_Optimization\\output\\demand_model.pkl']